Generador de dataset de numeros sintetico para entrenamiento de OCR

In [24]:
import os
import csv
import random
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

In [ ]:
# ----------------------------
# Config
# ----------------------------
OUT_DIR = Path(r"E:\Documents\Facultad\Proyecto\Data\dataset\synthetic_number2").resolve()
FONTS_DIR = Path(r"C:\Windows\Fonts\Arial\ariblk.ttf").resolve()  # Put .ttf here

N_SAMPLES = 1000
OUT_W, OUT_H = 64, 64

CHARSET = "0123456789"
MIN_LEN = 1
MAX_LEN = 1

P_OCCLUDE =0 
OUTPUT_BINARY = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [26]:
print(f"{OUT_DIR}")
print(f"{FONTS_DIR}")


E:\Documents\Facultad\Proyecto\Data\dataset\synthetic_number2
C:\Windows\Fonts\Arial\ariblk.ttf


In [27]:
# ----------------------------
# Utilities
# ----------------------------
def safe_mkdir(p: str):
    os.makedirs(p, exist_ok=True)

def list_ttf_fonts(folder: str):
    fonts = []
    for p in Path(folder).rglob("*"):
        if p.suffix.lower() in {".ttf", ".otf"}:
            fonts.append(str(p))
    return fonts

def rand_text():
    L = random.randint(MIN_LEN, MAX_LEN)
    return "".join(random.choice(CHARSET) for _ in range(L))

def add_background(h, w):
    """Creates a dark sign-like background with texture and gradient."""
    base = np.zeros((h, w), dtype=np.uint8)

    # Gradient
    if random.random() < 0.7:
        gx = np.linspace(0, random.randint(10, 60), w).astype(np.float32)
        gy = np.linspace(0, random.randint(10, 60), h).astype(np.float32)
        grad = (gy[:, None] + gx[None, :]) / 2.0
        base = np.clip(base.astype(np.float32) + grad, 0, 255).astype(np.uint8)

    # Noise texture
    noise_sigma = random.uniform(5, 25)
    noise = np.random.normal(0, noise_sigma, (h, w)).astype(np.float32)
    base = np.clip(base.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # Blur sometimes
    if random.random() < 0.35:
        k = random.choice([3, 5, 7])
        base = cv2.GaussianBlur(base, (k, k), sigmaX=random.uniform(0.5, 1.8))

    # Keep it dark
    base = (base * random.uniform(0.2, 0.6)).astype(np.uint8)
    return base

def draw_number_ttf(bg_gray: np.ndarray, text: str, font_paths: list[str]):
    """
    Draws white digits onto background using a TTF/OTF font (PIL),
    then returns a grayscale numpy image.
    """
    h, w = bg_gray.shape[:2]

    # Convert background to PIL image
    img = Image.fromarray(bg_gray, mode="L")
    draw = ImageDraw.Draw(img)

    font_path = random.choice(font_paths)

    # Choose font size: work canvas is bigger, so size can be large
    # Tune these ranges to match your crops
    if len(text) == 1:
        font_size = random.randint(int(h * 0.4), int(h * 0.6))
    else:
        font_size = random.randint(int(h * 0.4), int(h * 0.6))

    font = ImageFont.truetype(font_path, font_size)

    # Compute text bbox (PIL >= 8)
    bbox = draw.textbbox((0, 0), text, font=font, stroke_width=0)
    tw = bbox[2] - bbox[0]
    th = bbox[3] - bbox[1]

    # Center with jitter
    x = (w - tw) // 2 + random.randint(-10, 10)
    y = (h - th) // 2 + random.randint(-10, 10)

    # Clamp
    x = max(0, min(x, w - tw))
    y = max(0, min(y, h - th))

    # Stroke helps look like printed numbers
    #stroke_w = random.randint(0, 3)
    stroke_fill = 255

    # Draw white text
    draw.text(
        (x, y),
        text,
        fill=255,
        font=font,
        # stroke_width=0,
        # stroke_fill=stroke_fill,
    )

    out = np.array(img, dtype=np.uint8)

    # Optional morphology to simulate ink bleed / erosion
    if random.random() < 0.35:
        k = random.choice([1, 2])
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * k + 1, 2 * k + 1))
        if random.random() < 0.5:
            out = cv2.dilate(out, kernel, iterations=1)
        else:
            out = cv2.erode(out, kernel, iterations=1)

    return out

def random_perspective(img, strength=0.10):
    """Mild random perspective warp."""
    h, w = img.shape[:2]
    dx = int(w * strength)
    dy = int(h * strength)

    src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
    dst = np.float32([
        [random.randint(0, dx), random.randint(0, dy)],
        [w - 1 - random.randint(0, dx), random.randint(0, dy)],
        [w - 1 - random.randint(0, dx), h - 1 - random.randint(0, dy)],
        [random.randint(0, dx), h - 1 - random.randint(0, dy)],
    ])

    M = cv2.getPerspectiveTransform(src, dst)
    out = cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR,
                              borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out

def photometric(img):
    """Noise/blur/contrast variations."""
    out = img.astype(np.float32)

    alpha = random.uniform(0.8, 1.6)   # contrast
    beta = random.uniform(-15, 25)     # brightness
    out = out * alpha + beta

    if random.random() < 0.7:
        sigma = random.uniform(2, 18)
        out += np.random.normal(0, sigma, out.shape).astype(np.float32)

    out = np.clip(out, 0, 255).astype(np.uint8)

    if random.random() < 0.4:
        k = random.choice([3, 5])
        out = cv2.GaussianBlur(out, (k, k), sigmaX=random.uniform(0.3, 1.5))

    return out

def make_cable_polyline(w, h):
    """Creates a cable polyline crossing the image."""
    n_ctrl = random.randint(3, 5)

    def point_on_side(side):
        if side == "top":
            return (random.randint(0, w - 1), 0)
        if side == "bottom":
            return (random.randint(0, w - 1), h - 1)
        if side == "left":
            return (0, random.randint(0, h - 1))
        return (w - 1, random.randint(0, h - 1))

    s1 = random.choice(["top", "bottom", "left", "right"])
    s2 = random.choice(["top", "bottom", "left", "right"])
    while s2 == s1:
        s2 = random.choice(["top", "bottom", "left", "right"])

    pts = [point_on_side(s1)]
    for _ in range(n_ctrl - 2):
        pts.append((random.randint(0, w - 1), random.randint(0, h - 1)))
    pts.append(point_on_side(s2))
    pts = np.array(pts, dtype=np.int32)

    dense = []
    for i in range(len(pts) - 1):
        x0, y0 = pts[i]
        x1, y1 = pts[i + 1]
        steps = max(abs(x1 - x0), abs(y1 - y0), 1)
        for t in range(steps + 1):
            a = t / steps
            x = int(round((1 - a) * x0 + a * x1))
            y = int(round((1 - a) * y0 + a * y1))
            dense.append((x, y))

    dense = np.array(dense, dtype=np.int32)

    # Wiggle
    wiggle = random.uniform(0.5, 2.5)
    for i in range(0, len(dense), max(1, len(dense) // 25)):
        dense[i, 0] = np.clip(dense[i, 0] + int(random.uniform(-wiggle, wiggle)), 0, w - 1)
        dense[i, 1] = np.clip(dense[i, 1] + int(random.uniform(-wiggle, wiggle)), 0, h - 1)

    return dense.reshape(-1, 1, 2)

def add_cables(img):
    """Adds 1-3 cable-like occlusions."""
    out = img.copy()
    h, w = out.shape[:2]
    n_cables = random.choices([1, 2, 3], weights=[0.65, 0.25, 0.10])[0]

    for _ in range(n_cables):
        cable = make_cable_polyline(w, h)
        thickness = 1
        color = 0 if random.random() < 0.9 else 255

        cv2.polylines(out, [cable], isClosed=False, color=color,
                      thickness=thickness, lineType=cv2.LINE_AA)

        # if random.random() < 0.35:
        #     x, y = cable[random.randint(0, len(cable) - 1)][0]
        #     r = random.randint(1, 3)
        #     cv2.circle(out, (int(x), int(y)), r, color=color,
        #                thickness=-1, lineType=cv2.LINE_AA)

    return out

def binarize_like_pipeline(img):
    """Binary image, white digits on black."""
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Make background black
    if np.mean(th) > 127:
        th = 255 - th

    # if random.random() < 0.4:
    #     k = random.choice([1, 2])
    #     kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2 * k + 1, 2 * k + 1))
    #     th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel, iterations=1)

    return th
    
def crop_pad_resize(img: np.ndarray, out_w: int, out_h: int, pad_frac: float = 0.20) -> np.ndarray:
    """
    Finds foreground bbox, adds padding, then resizes to (out_w, out_h).
    Works for grayscale or binary where digits are brighter than background.
    """
    h, w = img.shape[:2]

    # Foreground mask (digits are bright)
    # Use a robust threshold based on percentile instead of fixed value
    t = np.percentile(img, 85)
    mask = (img > max(30, t)).astype(np.uint8) * 255

    # If mask is empty, just resize
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return cv2.resize(img, (out_w, out_h), interpolation=cv2.INTER_AREA)

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0 + 1
    bh = y1 - y0 + 1

    # Padding
    pad_x = int(bw * pad_frac)
    pad_y = int(bh * pad_frac)

    x0 = max(0, x0 - pad_x)
    y0 = max(0, y0 - pad_y)
    x1 = min(w - 1, x1 + pad_x)
    y1 = min(h - 1, y1 + pad_y)

    crop = img[y0:y1+1, x0:x1+1]

    # Letterbox to keep aspect ratio
    ch, cw = crop.shape[:2]
    scale = min(out_w / cw, out_h / ch)
    nw = max(1, int(round(cw * scale)))
    nh = max(1, int(round(ch * scale)))

    resized = cv2.resize(crop, (nw, nh), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((out_h, out_w), dtype=np.uint8)
    x = (out_w - nw) // 2
    y = (out_h - nh) // 2
    canvas[y:y+nh, x:x+nw] = resized
    return canvas

def random_projective_affine(
    img: np.ndarray,
    p_persp: float = 0.7,
    persp_strength: float = 0.12,
    rot_deg: float = 10.0,
    shear: float = 0.10,
    scale_jitter: float = 0.10,
    translate_frac: float = 0.06,
    border_value: int = 0,
) -> np.ndarray:
    """
    Applies a random combination of:
      - mild perspective warp
      - rotation
      - shear
      - scale and translation
    Designed for grayscale/binary digits on dark background.
    """
    h, w = img.shape[:2]
    out = img

    # --- Random affine (rot/scale/shear/translate) ---
    angle = random.uniform(-rot_deg, rot_deg)
    sc = random.uniform(1.0 - scale_jitter, 1.0 + scale_jitter)

    # Base rotation+scale
    M = cv2.getRotationMatrix2D((w * 0.5, h * 0.5), angle, sc)

    # Add shear (x and y)
    shx = random.uniform(-shear, shear)
    shy = random.uniform(-shear, shear)
    # Convert 2x3 to 3x3 to compose shear
    M3 = np.vstack([M, [0, 0, 1]]).astype(np.float32)
    S = np.array([[1, shx, 0],
                  [shy, 1, 0],
                  [0,  0, 1]], dtype=np.float32)
    A = S @ M3  # compose

    # Translation jitter
    tx = random.uniform(-translate_frac, translate_frac) * w
    ty = random.uniform(-translate_frac, translate_frac) * h
    A[0, 2] += tx
    A[1, 2] += ty

    out = cv2.warpAffine(
        out,
        A[:2, :],
        (w, h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=border_value,
    )

    # --- Optional perspective warp ---
    if random.random() < p_persp:
        dx = int(w * persp_strength)
        dy = int(h * persp_strength)

        src = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
        dst = np.float32([
            [random.randint(0, dx), random.randint(0, dy)],
            [w - 1 - random.randint(0, dx), random.randint(0, dy)],
            [w - 1 - random.randint(0, dx), h - 1 - random.randint(0, dy)],
            [random.randint(0, dx), h - 1 - random.randint(0, dy)],
        ])

        P = cv2.getPerspectiveTransform(src, dst)
        out = cv2.warpPerspective(
            out,
            P,
            (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=border_value,
        )

    return out


In [33]:
# ----------------------------
# Main
# ----------------------------
def main():
    # font_paths = list_ttf_fonts(FONTS_DIR)
    # if not font_paths:
    #     raise RuntimeError("No .ttf/.otf fonts found in FONTS_DIR. Put font files there.")

    img_dir = os.path.join(OUT_DIR, "images")
    safe_mkdir(img_dir)
    for c in CHARSET:
        safe_mkdir(Path(img_dir) / c)

    csv_path = os.path.join(OUT_DIR, "labels.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "label", "occluded", "font_file"])

        for i in range(N_SAMPLES):
            label = rand_text()

            # Work bigger -> downscale
            work_w, work_h = OUT_W, OUT_H
            bg = add_background(work_h, work_w)

            # Render with TTF
            font_used = FONTS_DIR
            img = draw_number_ttf(bg, label, [font_used])

            # Geometry + photometric
            if random.random() < 0.75:
                img = random_perspective(img, strength=random.uniform(0.03, 0.08))
           # img = photometric(img)
            if random.random() < 0.75:
                img = random_projective_affine(img)
           # img = photometric(img)
            occluded = 0
            if random.random() < P_OCCLUDE:
                img = add_cables(img)
                occluded = 1

            # Downscale to final
            img = cv2.resize(img, (OUT_W, OUT_H), interpolation=cv2.INTER_AREA)
            img = crop_pad_resize(img, OUT_W, OUT_H, pad_frac=0.25)

            if OUTPUT_BINARY:
                img = binarize_like_pipeline(img)

            out_name = f"{label}_{i:06d}.png"
            cv2.imwrite(os.path.join(img_dir,label, out_name), img)

            writer.writerow([f"images/{out_name}", label, occluded, Path(font_used).name])

            if (i + 1) % 2000 == 0:
                print(f"Generated {i+1}/{N_SAMPLES} samples")

    print("Done. Dataset written to:", OUT_DIR)

In [35]:
main()

C:\Users\Vale\AppData\Local\Temp\ipykernel_19820\2236332201.py:51: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(bg_gray, mode="L")


Generated 2000/3000 samples
Done. Dataset written to: E:\Documents\Facultad\Proyecto\Data\dataset\synthetic_number2
